# Path planning
This notebook implements spatial discovery pathfinding using A* Grid search and Ant Colony Optimization.

## 1. [A* search algorithm](https://www.geeksforgeeks.org/dsa/a-search-algorithm/)

The A* algorithm evaluates positions using the cost calculation function:

$$f(n) = g(n) + h(n)$$

Where $g(n)$ represents the exact path cost accrued from the origin node to target node $n$, and $h(n)$ represents the estimated heuristic distance remaining to reach the destination goal.

Heuristic Definitions:

Manhattan Distance: Suitable when movement is restricted to 4 directions (orthogonal).

$$h(n) = |x_1 - x_2| + |y_1 - y_2|$$

Euclidean Distance: Suitable when movement can happen at any angle (straight line distance).

$$h(n) = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2}$$

If an obstacle acts as a wall between two points, a Manhattan heuristic can over- or under-estimate the actual cost compared to a Euclidean heuristic. This discrepancy forces the algorithm to check different paths and alter the configuration of the final trajectory.

1. Implement the different heuristics in [`get_heuristic`](./PathPlanning/Optimizers/AStarOptimizer.py) (PathPlanning/Optimizers/AStarOptimizer.py Line n° 8) method.
2. Run the simulation multiple times. Pay attention to the path generated depending on the chosen heuristic. Is there an explanation ?

In [ ]:
import heapq
import matplotlib.pyplot as plt
import numpy as np

from PathPlanning.WorldGenerators import ClusterWorldGenerator
from PathPlanning.Optimizers import AStarOptimizer

a_star = AStarOptimizer()

# Create a map layout containing an obstacle corridor
grid_size = (20, 20)
start_pos = (10, 2)
goal_pos = (10, 16)

grid_map = ClusterWorldGenerator.generate(
    shape=grid_size,
    coverage=0.40,
    num_seeds=5,  # 5 distinct obstacle groupings
    start=start_pos,
    goal=goal_pos,
)

# Execute searches with different configurations
path_manhattan = a_star.optimize(grid_map, start_pos, goal_pos, "manhattan")
path_euclidean = a_star.optimize(grid_map, start_pos, goal_pos, "euclidean")

# Plotting Function
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
for i, (path, mode) in enumerate(
    zip([path_manhattan, path_euclidean], ["manhattan", "euclidean"])
):
    ax[i].imshow(grid_map, cmap="binary")
    if path:
        px, py = zip(*path)
        ax[i].plot(py, px, color="red", linewidth=3, label="Found Path")
    ax[i].scatter(start_pos[1], start_pos[0], color="green", label="Start", s=100)
    ax[i].scatter(goal_pos[1], goal_pos[0], color="blue", label="Goal", s=100)
    ax[i].set_title(f"Heuristic Exploration Mode: {mode.upper()}")
    ax[i].legend()
plt.show()

## 2. [Ant Colony Optimization](https://en.wikipedia.org/wiki/Ant_colony_optimization_algorithms)
See also [Introduction to Ant Colony Optimization](https://www.geeksforgeeks.org/machine-learning/introduction-to-ant-colony-optimization/).

Ant Colony Optimization (ACO) mimics how real ants find paths using pheromone trails. The probability of choosing a path depends on its pheromone intensity ($\tau$) and its visibility/closeness ($\eta = 1/d$):

$$P_{ij} = \frac{(\tau_{ij})^\alpha \cdot (\eta_{ij})^\beta}{\sum (\tau_{ik})^\alpha \cdot (\eta_{ik})^\beta}$$

Pheromones degrade over time due to evaporation, which helps prevent the algorithm from getting stuck in a local minimum:$$\tau_{ij} \leftarrow (1 - \rho)\tau_{ij} + \Delta \tau_{ij}$$

1. Change `alpha` value, what's its influence ?
2. Change `beta` value, what's its influence ?
3. Change `evaporation_rate` value, what's its influence ?
4. Change `num_ants` value, what's its influence ?

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import numpy as np

# User Imports
from PathPlanning.WorldGenerators import ClusterWorldGenerator
from PathPlanning.Optimizers.GridAntColonyOptimizer import GridAntColonyOptimizer

# =====================================================================
# 2. Environment Setup & Generation
# =====================================================================
grid_size = (20, 20)
start_pos = (10, 2)
goal_pos = (10, 16)

# Generate map using ClusterWorldGenerator
grid_map = ClusterWorldGenerator.generate(
    shape=grid_size,
    coverage=0.35, # Dropped slightly to guarantee paths through clusters
    num_seeds=5,
    start=start_pos,
    goal=goal_pos
)

# Instantiate the Optimizer
# Base configuration: alpha=1.2, beta=2.5, evaporation_rate=0.2
aco = GridAntColonyOptimizer(grid_map, alpha=1.2, beta=2.5, evaporation_rate=0.2)

# =====================================================================
# 3. Matplotlib Live Animation Construction
# =====================================================================
fig, ax = plt.subplots(figsize=(8, 7))

# Initialize base background canvas elements
ax.imshow(grid_map, cmap="binary", origin="upper", alpha=0.3)
pheromone_layer = ax.imshow(aco.get_spatial_intensity(), cmap="YlOrRd", origin="upper", alpha=0.7)
start_marker = ax.scatter(start_pos[1], start_pos[0], color="green", s=150, label="Start", zorder=5)
goal_marker = ax.scatter(goal_pos[1], goal_pos[0], color="blue", s=150, label="Goal", zorder=5)

ax.set_title("Iteration: 0 | Pheromone Distribution Matrix")
fig.colorbar(pheromone_layer, ax=ax, label="Pheromone Concentration")
ax.legend(loc="upper left")
ax.grid(color="gray", linestyle=":", linewidth=0.5)

def animate(frame):
    # Execute 1 step/generation of the colony
    paths, costs = aco.simulate_ants(start_pos, goal_pos, num_ants=15)
    if paths:
        aco.update_pheromones(paths, costs)

    # Refresh raw heatmap display array
    spatial_data = aco.get_spatial_intensity()
    pheromone_layer.set_array(spatial_data)

    # Dynamically scale visibility constraints
    pheromone_layer.set_clim(vmin=spatial_data.min(), vmax=spatial_data.max())

    ax.set_title(f"Colony Exploration Iteration: {frame + 1} \n Number of ants that found a path: {len(paths)}")
    return [pheromone_layer]

# Compile frames to render animation cleanly
anim = animation.FuncAnimation(fig, animate, frames=100, interval=250, blit=True)
plt.close() # Prevents static duplication showing up underneath loop window

# Display within Jupyter Notebook output cells
HTML(anim.to_jshtml())